<a href="https://colab.research.google.com/github/josethomas568/Salvage-Vehicle-Damage-Classification-CNN-/blob/main/Salvage_Vehicle_Damage_Classification(CNN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Salvage Vehicle Damage Classification(CNN)

~ Jose Thomas


In [2]:
import kagglehub
import os

# 1. Download the dataset
print("Downloading dataset...")
path = kagglehub.dataset_download("hamzamanssor/car-damage-assessment")

# 2. Locate the main image directory
print(f"Dataset cached at: {path}")

# List what is inside to verify the folder structure (e.g., 'data1a', 'data2a', etc.)
print("Contents of the dataset:")
print(os.listdir(path))

100%|██████████| 16.0M/16.0M [00:00<00:00, 46.2MB/s]

Extracting files...


Dataset cached at: /root/.cache/kagglehub/datasets/hamzamanssor/car-damage-assessment/versions/1
Contents of the dataset:
['image', 'data.csv']


In [11]:
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define paths
csv_path = f"{path}/data.csv"
base_dir = path
df = pd.read_csv(csv_path)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 2. Create the Data Generators WITHOUT rescaling
train_datagen = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(validation_split=0.2)

# 3. Training Pipeline
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    directory=base_dir,
    x_col="image",
    y_col="classes",
    subset="training",
    batch_size=BATCH_SIZE,
    seed=123,
    class_mode="categorical",
    target_size=IMG_SIZE
)

# 4. Validation Pipeline
val_generator = val_datagen.flow_from_dataframe(
    dataframe=df,
    directory=base_dir,
    x_col="image",
    y_col="classes",
    subset="validation",
    batch_size=BATCH_SIZE,
    seed=123,
    class_mode="categorical",
    target_size=IMG_SIZE
)

Found 1276 validated image filenames belonging to 8 classes.
Found 318 validated image filenames belonging to 8 classes.


In [12]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# 1. Load the pre-trained EfficientNetB0
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# 2. Freeze the base model so we don't destroy the pre-trained weights
base_model.trainable = False

# 3. Build our custom top layers for the 8 damage classes
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
predictions = Dense(8, activation='softmax')(x)
# 4. Stitch it all together into a final model
model = Model(inputs=base_model.input, outputs=predictions)

# 5. Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Architecture Compiled Successfully!")

Model Architecture Compiled Successfully!


In [13]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Stop training if validation accuracy doesn't improve for 5 epochs
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath='best_vehicle_damage_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_list = [early_stop, checkpoint]

In [14]:
# the training process
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks_list
)

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3175 - loss: 1.8747
Epoch 1: val_accuracy improved from None to 0.64465, saving model to best_vehicle_damage_model.keras

Epoch 1: finished saving model to best_vehicle_damage_model.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 138s 3s/step - accuracy: 0.4475 - loss: 1.5978 - val_accuracy: 0.6447 - val_loss: 1.0697
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6706 - loss: 1.0739
Epoch 2: val_accuracy improved from 0.64465 to 0.75157, saving model to best_vehicle_damage_model.keras

Epoch 2: finished saving model to best_vehicle_damage_model.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 120s 3s/step - accuracy: 0.6850 - loss: 1.0158 - val_accuracy: 0.7516 - val_loss: 0.7969
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7473 - loss: 0.8163
Epoch 3: val_accuracy improved from 0.75157 to 0.80818, saving model to best_vehicle_damage_model.keras

Epoch 3: finished saving model to best_vehicle_damage_model.keras
40/40 ━━━

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# 1. Get the exact damage categories the model learned
class_labels = list(train_generator.class_indices.keys())

# 2. Load your random test image
test_image_path = 'test_crash.jpg'

img = image.load_img(test_image_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)

# 3. Fire the inference engine
predictions = model.predict(img_array)[0]

# 4. Format the output into a clean dictionary
results = {label: float(prob) for label, prob in zip(class_labels, predictions)}
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)

print("\n --- DAMAGE ASSESSMENT REPORT --- ")
for label, prob in sorted_results[:3]: # Display the top 3 highest probabilities
    print(f"{label.upper()}: {prob * 100:.2f}% confidence")

In [16]:
import plotly.figure_factory as ff
from sklearn.metrics import confusion_matrix
import numpy as np

print("Running validation predictions... (This may take a minute)")

# 1. Reset the generator and predict on the entire validation dataset
val_generator.reset()
predictions = model.predict(val_generator)

# 2. Get the highest probability prediction for each image
y_pred = np.argmax(predictions, axis=1)

# 3. Get the actual true labels
y_true = val_generator.classes
class_names = list(val_generator.class_indices.keys())

# 4. Calculate the raw confusion matrix math
cm = confusion_matrix(y_true, y_pred)

# 5. Render the Interactive Plotly Heatmap
fig = ff.create_annotated_heatmap(
    z=cm,
    x=class_names, # Predicted labels on the bottom
    y=class_names, # Actual labels on the side
    colorscale='Blues',
    showscale=True
)

fig.update_layout(
    title="CNN Evaluation: Vehicle Damage Confusion Matrix",
    xaxis_title="Predicted Damage Classification",
    yaxis_title="Actual Damage Classification",
    font=dict(family="Arial", size=12),
    width=900,
    height=800,
    margin=dict(l=150, b=150) # Adds padding so the labels don't get cut off
)

# Render the interactive dashboard right inside Colab!
fig.show()

Running validation predictions... (This may take a minute)
10/10 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step
